## load library yang di perlukan dlu

In [1]:
import os
import json
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

## cek gpu cuda

In [2]:
import torch
print("Versi Torch:", torch.__version__)
print("Versi CUDA di Torch:", torch.version.cuda)
print("CUDA tersedia:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Versi Torch: 2.10.0+cu130
Versi CUDA di Torch: 13.0
CUDA tersedia: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## konfigurasi path

In [7]:
TRAIN_DIR = '../dataset/train'
VAL_DIR = '../dataset/val'
TEST_DIR = '../dataset/test'

MODEL_SAVE_PATH = 'best_model.pth'
CLASS_NAMES_PATH = 'class_names.json'

BATCH_SIZE = 32
EPOCH = 10
LEARNIG_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Di train menggunakan {DEVICE}")

Di train menggunakan cuda


## data preprocessing dan augmentasi

In [8]:
# untuk train di augmented dan di rotasi acek supaya model lebih bagus
train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# untuk val test ga augmented, biat pake data ori
val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## setelah kita buat augmentasi dan preprosecing kita bakalan load dataset kita, masukkan ke augmentasi kita tadi

In [10]:
print("loading datasets")
train_dataset = datasets.ImageFolder(TRAIN_DIR, transform=train_transforms)
val_dataset = datasets.ImageFolder(VAL_DIR, transform=val_test_transforms)
test_dataset = datasets.ImageFolder(TEST_DIR, transform=val_test_transforms)

# kita siapkan class names dalam json untuk du pakai be fast api kita nnatinya
class_names = train_dataset.classes
with open(CLASS_NAMES_PATH, 'w') as f:
    json.dump(class_names, f)
print(f"tersimpan {len(class_names)}, kelas di {CLASS_NAMES_PATH}")


loading datasets
tersimpan 10, kelas di class_names.json


## buat dataloader

In [11]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
test_loader =DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)

## setup arsitektur CNN dan Model MobileNetv3

In [14]:
print("setup model mobilenetv2...")
weghts = models.MobileNet_V2_Weights.DEFAULT
model = models.mobilenet_v2(weights=weghts)

#Ubah layer klasifikasi terakhir sesuai jumlah kelas penyakit
num_filter = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_filter, len(class_names))

model = model.to(DEVICE)

# lodd dan optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNIG_RATE)


setup model mobilenetv2...


## trainin loop

In [ ]:
best_val_acc = 0.0
for epoch in range(EPOCH):
    print(f"\n epoch {epoch+1}/{EPOCH}")
    print("-" * 15)

    model.train()
    running_loss, running_corrects = 0.0, 0